# DensGNN/DeepDensGNN — Euclidiana × MRD/MMSRD final

Este notebook reconstrói a última iteração de uma run a partir dos arquivos de embeddings e metadados. Ele mantém **os mesmos clusters HDBSCAN, pontos de suporte e pesos** e compara:

\[
d_E(i,j)=\sum_{s\in\Omega_j}w_{js}\lVert z_i-z_s\rVert_2
\]

com a distância usada pelo modelo:

- **DensGNN:** MRD com o mrd_k fixo;
- **DeepDensGNN:** MMSRD condicionada ao cluster, usando os candidate_k e pesos finais pi salvos.

A comparação é pós-treinamento: mostra como as duas distâncias se comportam no embedding final, mas não substitui uma ablação em que cada alternativa é treinada desde o início.

Limites importantes:

1. Os IDs dos suportes não são persistidos; eles são reconstruídos executando HDBSCAN novamente. O notebook valida a reconstrução contra assignments.npy.
2. Uma run **euclidiana do DeepDensGNN** não possui pi aprendido. Nesse caso não existe uma MMSRD final treinada para reconstruir, e o notebook interrompe a análise em vez de inventar pesos.
3. O cálculo forma uma matriz \(N\times N\), assim como o modelo, e pode consumir bastante memória.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def encontrar_raiz(inicio=None):
    inicio = Path(inicio or Path.cwd()).resolve()
    for pasta in (inicio, *inicio.parents):
        if (pasta / "models/DensGNN/src").is_dir() and (pasta / "scripts").is_dir():
            return pasta
    raise FileNotFoundError("Abra o notebook dentro deste repositório.")

RAIZ = encontrar_raiz()
sys.path.insert(0, str(RAIZ / "models/DensGNN/src"))
sys.path.insert(0, str(RAIZ / "models/DeepDensGNN"))

from graph_benchmark.models.densgnn import DensGNNConfig, _build_cluster_state
from graph_benchmark.utils.densgnn_loss import (
    _kth_neighbor_distances,
    support_cluster_distances,
)
from density import cluster_conditioned_distances, multiscale_core_distances

# Pode ser a pasta densgnn/deep_densgnn ou o próprio embeddings.npy.
ARTEFATO = (
    RAIZ
    # / "results/cora/runs/20260922_025611_872125_seed2075_run50/deep_densgnn"
    / "results/cora_final/runs/20261008_203228_010862_seed2027_run02/deep_densgnn"
)
METADATA_YAML = None
ASSIGNMENTS_NPY = None
MAX_PARES_GRAFICO = 2_000_000
MAX_PARES_SUPORTE_GRAFICO = 500_000
PERCENTIL_LIMITE_GRAFICO = 99.5
SEED_GRAFICO = 42
EXIGIR_REPRODUCAO_EXATA = True
SALVAR_RESULTADOS = False
PASTA_SAIDA = None

def resolver(caminho):
    caminho = Path(caminho).expanduser()
    return caminho.resolve() if caminho.is_absolute() else (RAIZ / caminho).resolve()

In [ ]:
artefato = resolver(ARTEFATO)
arquivo_embedding = artefato if artefato.is_file() else artefato / "embeddings.npy"
pasta_modelo = arquivo_embedding.parent
arquivo_metadata = resolver(METADATA_YAML) if METADATA_YAML else pasta_modelo / "metadata.yaml"
arquivo_assignments = resolver(ASSIGNMENTS_NPY) if ASSIGNMENTS_NPY else pasta_modelo / "assignments.npy"
arquivo_metrics = pasta_modelo / "metrics.json"

for arquivo in (arquivo_embedding, arquivo_metadata):
    if not arquivo.is_file():
        raise FileNotFoundError(arquivo)

embeddings_np = np.load(arquivo_embedding, allow_pickle=False)
if embeddings_np.ndim != 2 or embeddings_np.shape[0] < 3:
    raise ValueError(f"Esperava embeddings [nós, dimensões], recebi {embeddings_np.shape}.")
if not np.issubdtype(embeddings_np.dtype, np.number) or not np.isfinite(embeddings_np).all():
    raise ValueError("Os embeddings precisam ser numéricos e finitos.")
embeddings = torch.as_tensor(np.asarray(embeddings_np, dtype=np.float32))

with arquivo_metadata.open(encoding="utf-8") as arquivo:
    metadata = yaml.safe_load(arquivo) or {}
algorithm_metadata = {}
if arquivo_metrics.is_file():
    algorithm_metadata = json.loads(
        arquivo_metrics.read_text(encoding="utf-8")
    ).get("algorithm") or {}

modelo = str(metadata.get("model") or algorithm_metadata.get("name") or pasta_modelo.name).lower()
if modelo not in {"densgnn", "deep_densgnn"}:
    raise ValueError(f"Modelo não suportado: {modelo!r}.")
parametros = dict(metadata.get("model_params") or {})
if not parametros:
    parametros = dict(algorithm_metadata)

necessarios = (
    "point_selection", "point_probability_threshold",
    "hdbscan_min_cluster_size", "hdbscan_min_samples",
    "hdbscan_cluster_selection_method", "mrd_k",
)
faltantes = [campo for campo in necessarios if parametros.get(campo) is None]
if faltantes:
    raise ValueError(f"Parâmetros ausentes no metadata: {faltantes}")

modo_treinado = str(parametros.get("support_distance", "mrd")).lower()
config = DensGNNConfig(
    input_channels=embeddings.shape[1],
    hidden_channels=embeddings.shape[1],
    embedding_channels=embeddings.shape[1],
    point_selection=str(parametros["point_selection"]),
    point_probability_threshold=float(parametros["point_probability_threshold"]),
    hdbscan_min_cluster_size=int(parametros["hdbscan_min_cluster_size"]),
    hdbscan_min_samples=int(parametros["hdbscan_min_samples"]),
    hdbscan_cluster_selection_method=str(parametros["hdbscan_cluster_selection_method"]),
    mrd_k=int(parametros["mrd_k"]),
    support_distance=modo_treinado,
)
with torch.no_grad():
    estado = _build_cluster_state(embeddings, config)
if estado is None:
    raise RuntimeError("A reconstrução HDBSCAN produziu menos de dois clusters válidos.")

print("Modelo:", modelo)
print("Distância usada no treinamento:", modo_treinado)
print("Embedding:", arquivo_embedding)
print("Formato:", tuple(embeddings.shape))
print("Clusters/suportes reconstruídos:", len(estado.supports))
print("Labels HDBSCAN:", [int(s.label) for s in estado.supports])

In [ ]:
with torch.no_grad():
    dist_euclid = support_cluster_distances(
        embeddings,
        estado.supports,
        mrd_k=config.mrd_k,
        support_distance="euclidean",
    )

    if modelo == "densgnn":
        nome_distancia = "MRD"
        dist_mrd = support_cluster_distances(
            embeddings,
            estado.supports,
            mrd_k=config.mrd_k,
            support_distance="mrd",
        )
        core_node = _kth_neighbor_distances(
            torch.cdist(embeddings, embeddings, p=2), config.mrd_k
        )
        core_por_cluster = core_node[:, None].expand(-1, len(estado.supports))
        k_eff_ordenado = [float(config.mrd_k)] * len(estado.supports)
        descricao_mrd = f"MRD de k único (k={config.mrd_k})"

    else:
        nome_distancia = "MMSRD"
        info_escalas = (
            metadata.get("density_scale_info")
            or algorithm_metadata.get("density_scale_info")
            or {}
        )
        if not info_escalas.get("active", False) or not info_escalas.get("pi"):
            raise RuntimeError(
                "Esta run do DeepDensGNN não contém pesos pi ativos. "
                "Isso é esperado em runs treinadas com support_distance=euclidean. "
                "Use uma run MRD para reconstruir a MMSRD aprendida."
            )

        candidate_k = tuple(int(k) for k in info_escalas["candidate_k"])
        labels_salvos = [int(label) for label in info_escalas["cluster_labels"]]
        pi_salvo = np.asarray(info_escalas["pi"], dtype=np.float32)
        k_eff_salvo = np.asarray(info_escalas["k_eff"], dtype=np.float32)
        if pi_salvo.shape != (len(labels_salvos), len(candidate_k)):
            raise ValueError("Formato de pi incompatível com cluster_labels/candidate_k.")
        if len(k_eff_salvo) != len(labels_salvos):
            raise ValueError("k_eff não está alinhado com cluster_labels.")

        indice_label = {label: i for i, label in enumerate(labels_salvos)}
        labels_reconstruidos = [int(s.label) for s in estado.supports]
        ausentes = sorted(set(labels_reconstruidos) - set(indice_label))
        if ausentes:
            raise RuntimeError(
                f"Clusters reconstruídos ausentes nos pesos pi salvos: {ausentes}. "
                "O HDBSCAN não foi reproduzido exatamente."
            )
        ordem = [indice_label[label] for label in labels_reconstruidos]
        pi = torch.as_tensor(pi_salvo[ordem], dtype=embeddings.dtype)
        k_eff_ordenado = k_eff_salvo[ordem].astype(float).tolist()

        core_multiescala, pairwise = multiscale_core_distances(embeddings, candidate_k)
        dist_mrd = cluster_conditioned_distances(
            pairwise, core_multiescala, estado.supports, pi
        )
        core_por_cluster = core_multiescala @ pi.T
        core_node = core_por_cluster.mean(dim=1)
        descricao_mrd = f"MMSRD ({len(candidate_k)} escalas candidatas)"

dist_euclid = dist_euclid.cpu().numpy()
dist_mrd = dist_mrd.cpu().numpy()
core_node = core_node.cpu().numpy()
core_por_cluster = core_por_cluster.cpu().numpy()

pred_euclid = np.argmin(dist_euclid, axis=1)
pred_mrd = np.argmin(dist_mrd, axis=1)
mudou = pred_euclid != pred_mrd
delta = dist_mrd - dist_euclid
tolerancia = 1e-4 * np.maximum(1.0, np.abs(dist_euclid))
violacoes = int(np.count_nonzero(delta < -tolerancia))
if violacoes:
    raise RuntimeError(f"{nome_distancia} menor que Euclidiana em {violacoes} pares além da tolerância.")

print("Distância reconstruída:", descricao_mrd)
print(f"Pares nó–cluster: {delta.size:,}")
print(f"{nome_distancia} maior: {(delta > tolerancia).sum():,} ({(delta > tolerancia).mean():.1%})")
print(f"Delta mediano: {np.median(delta):.6g}; máximo: {np.max(delta):.6g}")
print(f"Nós cuja escolha muda: {mudou.sum():,}/{len(mudou):,} ({mudou.mean():.2%})")
print(f"ARI entre partições: {adjusted_rand_score(pred_mrd, pred_euclid):.6f}")
print(f"NMI entre partições: {normalized_mutual_info_score(pred_mrd, pred_euclid):.6f}")

In [ ]:
atribuicoes_salvas = None
if arquivo_assignments.is_file():
    atribuicoes_salvas = np.load(arquivo_assignments, allow_pickle=False).reshape(-1)
    if len(atribuicoes_salvas) != len(pred_mrd):
        raise ValueError("assignments.npy e embeddings.npy têm quantidades diferentes de nós.")
    pred_run = pred_mrd if modo_treinado == "mrd" else pred_euclid
    divergentes = int(np.count_nonzero(atribuicoes_salvas != pred_run))
    mensagem = (
        f"Validação da distância treinada ({modo_treinado}): "
        f"{len(pred_run) - divergentes}/{len(pred_run)} atribuições reproduzidas; "
        f"{divergentes} divergentes."
    )
    if divergentes and EXIGIR_REPRODUCAO_EXATA:
        raise RuntimeError(
            mensagem
            + " A reconstrução dos suportes não é idêntica; confira versão do código, "
              "dependências e artefatos da run."
        )
    print(mensagem)
else:
    warnings.warn("assignments.npy não encontrado: a reconstrução não pôde ser validada.")

estatisticas = []
for coluna, suporte in enumerate(estado.supports):
    estatisticas.append({
        "cluster_label": int(suporte.label),
        "support_points": int(suporte.node_ids.numel()),
        "alpha_j": estado.selection_alphas.get(int(suporte.label)),
        "relative_density": estado.relative_densities.get(int(suporte.label)),
        "k_eff": k_eff_ordenado[coluna],
        "distance_name": nome_distancia,
        "assigned_density_distance": int(np.count_nonzero(pred_mrd == coluna)),
        "assigned_euclidean": int(np.count_nonzero(pred_euclid == coluna)),
        "mean_euclidean": float(dist_euclid[:, coluna].mean()),
        "mean_density_distance": float(dist_mrd[:, coluna].mean()),
        "mean_delta": float(delta[:, coluna].mean()),
    })
tabela_clusters = pd.DataFrame(estatisticas)
display(tabela_clusters.round(5))

In [ ]:
def margem_duas_melhores(distancias):
    duas = np.partition(distancias, 1, axis=1)[:, :2]
    duas.sort(axis=1)
    return duas[:, 1] - duas[:, 0]

rng = np.random.default_rng(SEED_GRAFICO)
total_pares = dist_euclid.size
indices = rng.choice(total_pares, size=min(MAX_PARES_GRAFICO, total_pares), replace=False)
linhas, colunas = np.unravel_index(indices, dist_euclid.shape)
destaque = mudou[linhas]
margem_e = margem_duas_melhores(dist_euclid)
margem_m = margem_duas_melhores(dist_mrd)

fig, eixos = plt.subplots(1, 3, figsize=(18, 5.2), constrained_layout=True)

pontos = eixos[0].scatter(
    dist_euclid[linhas, colunas],
    dist_mrd[linhas, colunas],
    c=np.log1p(core_node[linhas]),
    cmap="viridis", s=9, alpha=0.5, rasterized=True,
)
if destaque.any():
    eixos[0].scatter(
        dist_euclid[linhas[destaque], colunas[destaque]],
        dist_mrd[linhas[destaque], colunas[destaque]],
        facecolors="none", edgecolors="crimson", linewidths=0.6, s=28,
        label="Nó cuja escolha mudou",
    )
limite_min = float(min(dist_euclid.min(), dist_mrd.min()))
limite_max = float(max(dist_euclid.max(), dist_mrd.max()))
eixos[0].plot([limite_min, limite_max], [limite_min, limite_max], "k--", lw=1, label="y = x")
eixos[0].set(
    title="Cada ponto = par nó–cluster",
    xlabel="Distância euclidiana aos suportes",
    ylabel=f"Distância {nome_distancia} aos mesmos suportes",
)
eixos[0].legend()
fig.colorbar(pontos, ax=eixos[0], label="log(1 + raio local do nó)")

delta_flat = delta.ravel()
tolerancia_flat = tolerancia.ravel()
eixos[1].hist(delta_flat, bins=60, color="slateblue", alpha=0.8)
eixos[1].axvline(0, color="black", linestyle="--", linewidth=1)
eixos[1].set(
    title=f"{nome_distancia} − Euclidiana\nzeros: {np.mean(np.abs(delta_flat) <= tolerancia_flat):.1%}",
    xlabel="Aumento da distância", ylabel="Pares nó–cluster",
)

cores = np.where(mudou, "crimson", "steelblue")
eixos[2].scatter(margem_e, margem_m, c=cores, s=11, alpha=0.55, rasterized=True)
eixos[2].set(
    title=f"Margem entre os 2 clusters mais próximos\nmudança: {mudou.sum()}/{len(mudou)} nós",
    xlabel="Margem euclidiana", ylabel=f"Margem {nome_distancia}",
)
fig.suptitle(
    f"{modelo} | run {pasta_modelo.parent.name} | treinado com {modo_treinado}",
    fontsize=14,
)

if SALVAR_RESULTADOS:
    pasta_saida = resolver(PASTA_SAIDA) if PASTA_SAIDA else pasta_modelo / "distance_comparison"
    pasta_saida.mkdir(parents=True, exist_ok=True)
    sufixo_distancia = nome_distancia.lower()
    fig.savefig(pasta_saida / f"euclidean_vs_{sufixo_distancia}.png", dpi=180, bbox_inches="tight")
    tabela_clusters.to_csv(pasta_saida / "cluster_distance_summary.csv", index=False)
    pd.DataFrame({
        "node_id": np.arange(len(pred_mrd)),
        "cluster_density_distance": pred_mrd,
        "cluster_euclidean": pred_euclid,
        "changed": mudou,
        "margin_mrd": margem_m,
        "margin_euclidean": margem_e,
    }).to_csv(pasta_saida / "node_distance_summary.csv", index=False)
    print("Resultados salvos em:", pasta_saida)
plt.show()

## Comparação direta ponto–suporte

A figura anterior compara as distâncias depois da agregação por cluster. A próxima figura examina a definição elementar da distância: para cada suporte s do cluster j, compara a distância euclidiana com MRD ou MMSRD_j(i,s). A amostragem é feita sobre todos os pares nó–suporte sem construir uma nova matriz tridimensional.

In [ ]:
def amostrar_pares_no_suporte(
    embeddings_array,
    supports,
    effective_core,
    max_pairs,
    seed,
):
    n_nodes = embeddings_array.shape[0]
    support_sizes = np.asarray(
        [int(support.node_ids.numel()) for support in supports], dtype=np.int64
    )
    pair_counts = n_nodes * support_sizes
    cumulative = np.cumsum(pair_counts)
    total = int(cumulative[-1])
    sample_size = total if max_pairs is None else min(int(max_pairs), total)
    rng_local = np.random.default_rng(seed)
    selected = rng_local.choice(total, size=sample_size, replace=False)
    cluster_columns = np.searchsorted(cumulative, selected, side="right")

    euclidean_parts = []
    density_parts = []
    dominant_parts = []
    cluster_parts = []
    for column in np.unique(cluster_columns):
        mask = cluster_columns == column
        start = 0 if column == 0 else int(cumulative[column - 1])
        local = selected[mask] - start
        support_ids = supports[column].node_ids.detach().cpu().numpy()
        support_size = support_ids.size
        node_ids = local // support_size
        support_positions = local % support_size
        selected_support_ids = support_ids[support_positions]

        euclidean = np.linalg.norm(
            embeddings_array[node_ids] - embeddings_array[selected_support_ids],
            axis=1,
        )
        query_core = effective_core[node_ids, column]
        support_core = effective_core[selected_support_ids, column]
        components = np.stack((euclidean, query_core, support_core), axis=0)
        density_distance = components.max(axis=0)
        dominant = components.argmax(axis=0)

        euclidean_parts.append(euclidean.astype(np.float32, copy=False))
        density_parts.append(density_distance.astype(np.float32, copy=False))
        dominant_parts.append(dominant.astype(np.int8, copy=False))
        cluster_parts.append(
            np.full(mask.sum(), int(supports[column].label), dtype=np.int64)
        )

    return {
        "euclidean": np.concatenate(euclidean_parts),
        "density": np.concatenate(density_parts),
        "dominant": np.concatenate(dominant_parts),
        "cluster_label": np.concatenate(cluster_parts),
        "total_available": total,
    }

amostra_suporte = amostrar_pares_no_suporte(
    embeddings_np,
    estado.supports,
    core_por_cluster,
    MAX_PARES_SUPORTE_GRAFICO,
    SEED_GRAFICO,
)
par_euclid = amostra_suporte["euclidean"]
par_densidade = amostra_suporte["density"]
par_delta = par_densidade - par_euclid
par_tolerancia = 1e-4 * np.maximum(1.0, np.abs(par_euclid))
par_ativo = par_delta > par_tolerancia
if np.any(par_delta < -par_tolerancia):
    raise RuntimeError(f"Amostra contém {nome_distancia} menor que a Euclidiana.")

nomes_componentes = np.asarray([
    "Euclidiana",
    "Core distance do nó",
    "Core distance do suporte",
])
contagens_componentes = np.bincount(
    amostra_suporte["dominant"], minlength=len(nomes_componentes)
)
percentuais_componentes = contagens_componentes / contagens_componentes.sum()

atividade_clusters = []
for support, k_eff in zip(estado.supports, k_eff_ordenado, strict=True):
    label = int(support.label)
    mask = amostra_suporte["cluster_label"] == label
    if mask.any():
        atividade_clusters.append({
            "cluster_label": label,
            "sampled_pairs": int(mask.sum()),
            "density_active_fraction": float(par_ativo[mask].mean()),
            "mean_increase": float(par_delta[mask].mean()),
            "k_eff": float(k_eff),
            "alpha_j": estado.selection_alphas.get(label),
        })
tabela_atividade_clusters = pd.DataFrame(atividade_clusters)
display(
    tabela_atividade_clusters.style.format({
        "density_active_fraction": "{:.2%}",
        "mean_increase": "{:.6g}",
        "k_eff": "{:.3f}",
        "alpha_j": "{:.3f}",
    })
)

fig_suporte, eixos_suporte = plt.subplots(
    2, 2, figsize=(14, 11), constrained_layout=True
)
limite_suporte = float(np.percentile(
    np.maximum(par_euclid, par_densidade), PERCENTIL_LIMITE_GRAFICO
))
dentro = (par_euclid <= limite_suporte) & (par_densidade <= limite_suporte)
mapa_suporte = eixos_suporte[0, 0].hexbin(
    par_euclid[dentro],
    par_densidade[dentro],
    gridsize=115,
    extent=(0, limite_suporte, 0, limite_suporte),
    mincnt=1,
    cmap="magma",
    norm=LogNorm(),
)
eixos_suporte[0, 0].plot(
    [0, limite_suporte], [0, limite_suporte], "c--", linewidth=1.2
)
eixos_suporte[0, 0].set(
    title=f"A. Distância elementar — até p{PERCENTIL_LIMITE_GRAFICO:g}",
    xlabel=r"$\|z_i-z_s\|_2$",
    ylabel=fr"$\mathrm{{{nome_distancia}}}_j(i,s)$",
    xlim=(0, limite_suporte),
    ylim=(0, limite_suporte),
)
eixos_suporte[0, 0].set_aspect("equal", adjustable="box")
fig_suporte.colorbar(
    mapa_suporte, ax=eixos_suporte[0, 0],
    label="Pares nó–suporte por hexágono (escala log)",
)

eixos_suporte[0, 1].hist(
    par_delta, bins=70, color="slateblue", alpha=0.85, log=True
)
eixos_suporte[0, 1].axvline(0, color="black", linestyle="--", linewidth=1)
eixos_suporte[0, 1].set(
    title=(
        f"B. {nome_distancia} − Euclidiana\n"
        f"distância de densidade ativa em {par_ativo.mean():.1%}"
    ),
    xlabel="Aumento da distância",
    ylabel="Pares nó–suporte (escala log)",
)

cores_componentes = ["#4c78a8", "#f58518", "#54a24b"]
eixos_suporte[1, 0].bar(
    nomes_componentes,
    100 * percentuais_componentes,
    color=cores_componentes,
)
eixos_suporte[1, 0].set(
    title=f"C. Termo dominante da {nome_distancia}",
    ylabel="Pares amostrados (%)",
)
eixos_suporte[1, 0].tick_params(axis="x", rotation=15)
for index, value in enumerate(100 * percentuais_componentes):
    eixos_suporte[1, 0].text(index, value, f"{value:.1f}%", ha="center", va="bottom")

labels_cluster = tabela_atividade_clusters["cluster_label"].astype(str).tolist()
valores_atividade = 100 * tabela_atividade_clusters["density_active_fraction"].to_numpy()
barras = eixos_suporte[1, 1].bar(
    labels_cluster,
    valores_atividade,
    color=plt.colormaps["viridis"](
        np.linspace(0.15, 0.9, len(labels_cluster))
    ),
)
eixos_suporte[1, 1].set(
    title=f"D. Ativação da {nome_distancia} por cluster",
    xlabel="Label HDBSCAN",
    ylabel="Pares em que a densidade domina (%)",
)
for barra, valor in zip(barras, valores_atividade, strict=True):
    eixos_suporte[1, 1].text(
        barra.get_x() + barra.get_width() / 2,
        valor,
        f"{valor:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
    )

fig_suporte.suptitle(
    f"Euclidiana × {nome_distancia} no nível ponto–suporte | "
    f"{len(par_euclid):,}/{amostra_suporte['total_available']:,} pares",
    fontsize=14,
)

if SALVAR_RESULTADOS:
    pasta_saida = resolver(PASTA_SAIDA) if PASTA_SAIDA else pasta_modelo / "distance_comparison"
    pasta_saida.mkdir(parents=True, exist_ok=True)
    fig_suporte.savefig(
        pasta_saida / f"euclidean_vs_{nome_distancia.lower()}_point_support.png",
        dpi=180,
        bbox_inches="tight",
    )
    tabela_atividade_clusters.to_csv(
        pasta_saida / "point_support_activity_by_cluster.csv", index=False
    )
plt.show()

## Análise agregada entre várias runs

A próxima seção repete a comparação sobre várias pastas de resultados. Cada run contribui com uma única observação, independentemente do número de nós ou clusters.

Por padrão, a busca considera apenas runs treinadas com MRD. Ajuste as raízes para selecionar outros datasets. O processamento reconstrói uma matriz de distâncias entre todos os pares de nós e pode levar alguns minutos em coleções grandes.

In [ ]:
pastas_agregadas = descobrir_pastas_modelo(
    RAIZES_RUNS_AGREGADAS, MODELOS_AGREGADOS
)
if LIMITE_RUNS_AGREGADAS is not None:
    pastas_agregadas = pastas_agregadas[: int(LIMITE_RUNS_AGREGADAS)]
if not pastas_agregadas:
    raise FileNotFoundError("Nenhuma pasta com embeddings.npy foi encontrada.")

resumos, ignoradas, amostras_heatmap = [], [], []
for indice, pasta in enumerate(pastas_agregadas, start=1):
    try:
        meta_previa, algo_previo = carregar_metadata_modelo(pasta)
        params_previos = dict(meta_previa.get("model_params") or algo_previo)
        modo_previo = str(params_previos.get("support_distance", "mrd")).lower()
        if (
            MODOS_TREINAMENTO_AGREGADOS is not None
            and modo_previo not in MODOS_TREINAMENTO_AGREGADOS
        ):
            ignoradas.append({
                "path": str(pasta),
                "reason": f"modo de treinamento filtrado: {modo_previo}",
            })
            continue
        print(f"[{indice}/{len(pastas_agregadas)}] {pasta.parent.name}/{pasta.name}")
        resumo_run, amostra_run = resumir_comparacao_run(pasta)
        resumos.append(resumo_run)
        amostras_heatmap.append(amostra_run)
    except Exception as erro:
        ignoradas.append({"path": str(pasta), "reason": str(erro)})
        warnings.warn(f"Run ignorada: {pasta}\n{erro}")

tabela_runs = pd.DataFrame(resumos)
if tabela_runs.empty:
    raise RuntimeError("Nenhuma run pôde ser analisada.")

colunas_exibicao = [
    "dataset", "run_id", "model", "seed", "n_nodes", "n_clusters",
    "distance_changed_fraction", "node_changed_fraction",
    "p95_relative_increase", "partition_ari", "partition_nmi",
]
display(
    tabela_runs[colunas_exibicao]
    .sort_values(["dataset", "model", "seed"])
    .reset_index(drop=True)
    .style.format({
        "distance_changed_fraction": "{:.2%}",
        "node_changed_fraction": "{:.2%}",
        "p95_relative_increase": "{:.2%}",
        "partition_ari": "{:.6f}",
        "partition_nmi": "{:.6f}",
    })
)
if ignoradas:
    print(f"Runs ignoradas: {len(ignoradas)}")
    display(pd.DataFrame(ignoradas))

In [ ]:
def boxplot_com_runs(ax, tabela, coluna, titulo, rotulo_y, multiplicador=100.0):
    modelos = sorted(tabela["model"].unique())
    valores = [
        tabela.loc[tabela["model"] == item, coluna].to_numpy() * multiplicador
        for item in modelos
    ]
    caixas = ax.boxplot(
        valores, positions=np.arange(len(modelos)), widths=0.55,
        patch_artist=True, showfliers=False, tick_labels=modelos,
        medianprops={"color": "black", "linewidth": 1.5},
    )
    for caixa in caixas["boxes"]:
        caixa.set(facecolor="lightgray", alpha=0.55)
    rng_local = np.random.default_rng(SEED_GRAFICO)
    for posicao, item in enumerate(modelos):
        parte = tabela[tabela["model"] == item]
        x = posicao + rng_local.normal(0, 0.045, size=len(parte))
        ax.scatter(
            x, parte[coluna].to_numpy() * multiplicador,
            c=[cores_dataset[d] for d in parte["dataset"]],
            edgecolors="white", linewidths=0.45,
            s=38, alpha=0.85, zorder=3,
        )
    ax.set(title=titulo, ylabel=rotulo_y)
    ax.grid(axis="y", alpha=0.22)

datasets_agregados = sorted(tabela_runs["dataset"].unique())
mapa_cores = plt.colormaps["tab10"]
cores_dataset = {
    nome: mapa_cores(indice % 10) for indice, nome in enumerate(datasets_agregados)
}
marcadores_modelo = {"densgnn": "o", "deep_densgnn": "^"}
nomes_distancia_agregados = sorted(tabela_runs["distance_name"].unique())
rotulo_distancia_agregado = "/".join(nomes_distancia_agregados)
sufixo_distancia_agregado = rotulo_distancia_agregado.lower().replace("/", "_")

fig_agregada, eixos_agregados = plt.subplots(
    2, 2, figsize=(14, 10), constrained_layout=True
)
boxplot_com_runs(
    eixos_agregados[0, 0], tabela_runs, "distance_changed_fraction",
    f"A. Pares nó–cluster afetados pela {rotulo_distancia_agregado}", "Pares alterados (%)",
)
boxplot_com_runs(
    eixos_agregados[0, 1], tabela_runs, "node_changed_fraction",
    "B. Vértices cuja atribuição muda", "Vértices alterados (%)",
)

for (dataset_item, modelo_item), parte in tabela_runs.groupby(["dataset", "model"]):
    eixos_agregados[1, 0].scatter(
        100 * parte["distance_changed_fraction"],
        100 * parte["node_changed_fraction"],
        color=cores_dataset[dataset_item],
        marker=marcadores_modelo.get(modelo_item, "o"),
        s=58, alpha=0.85, edgecolors="white", linewidths=0.5,
    )
eixos_agregados[1, 0].set(
    title="C. Alteração numérica × alteração da decisão",
    xlabel="Pares nó–cluster alterados (%)",
    ylabel="Vértices alterados (%)",
)
eixos_agregados[1, 0].grid(alpha=0.22)

boxplot_com_runs(
    eixos_agregados[1, 1], tabela_runs, "p95_relative_increase",
    "D. Intensidade da alteração", "Percentil 95 do aumento relativo (%)",
)
valores_p95 = 100 * tabela_runs["p95_relative_increase"].to_numpy()
if np.nanmax(valores_p95) > 1000:
    eixos_agregados[1, 1].set_yscale("symlog", linthresh=1)

legenda_datasets = [
    Line2D(
        [0], [0], marker="o", color="none", markerfacecolor=cores_dataset[nome],
        markeredgecolor="white", markersize=8, label=nome,
    )
    for nome in datasets_agregados
]
legenda_modelos = [
    Line2D(
        [0], [0], marker=marcadores_modelo.get(nome, "o"), color="black",
        linestyle="none", markersize=7, label=nome,
    )
    for nome in sorted(tabela_runs["model"].unique())
]
fig_agregada.legend(
    handles=legenda_datasets + legenda_modelos,
    loc="outside lower center",
    ncol=max(2, len(legenda_datasets + legenda_modelos)),
)
fig_agregada.suptitle(
    f"Euclidiana × {rotulo_distancia_agregado} no embedding final — {len(tabela_runs)} runs",
    fontsize=15,
)

if SALVAR_AGREGADO:
    pasta_agregada = resolver(PASTA_SAIDA_AGREGADA)
    pasta_agregada.mkdir(parents=True, exist_ok=True)
    tabela_runs.to_csv(pasta_agregada / "all_runs_distance_summary.csv", index=False)
    fig_agregada.savefig(
        pasta_agregada / f"all_runs_euclidean_vs_{sufixo_distancia_agregado}.png",
        dpi=180, bbox_inches="tight",
    )
    if ignoradas:
        pd.DataFrame(ignoradas).to_csv(
            pasta_agregada / "ignored_runs.csv", index=False
        )
    print("Resultados agregados salvos em:", pasta_agregada)
plt.show()

In [ ]:
if not amostras_heatmap:
    raise RuntimeError("Não há amostras disponíveis para construir o heatmap.")

euclid_heatmap = np.concatenate([item["euclidean"] for item in amostras_heatmap])
mrd_heatmap = np.concatenate([item["mrd"] for item in amostras_heatmap])
euclid_normalizada = np.concatenate([
    item["euclidean"] / item["scale"] for item in amostras_heatmap
])
mrd_normalizada = np.concatenate([
    item["mrd"] / item["scale"] for item in amostras_heatmap
])

def desenhar_heatmap_distancias(ax, x, y, titulo, rotulo):
    validos = np.isfinite(x) & np.isfinite(y) & (x >= 0) & (y >= 0)
    x, y = x[validos], y[validos]
    limite = float(np.percentile(np.maximum(x, y), PERCENTIL_LIMITE_HEATMAP))
    if not np.isfinite(limite) or limite <= 0:
        raise ValueError("Não foi possível determinar os limites do heatmap.")
    dentro = (x <= limite) & (y <= limite)
    mapa = ax.hexbin(
        x[dentro], y[dentro],
        gridsize=115, extent=(0, limite, 0, limite),
        mincnt=1, cmap="magma", norm=LogNorm(),
    )
    ax.plot([0, limite], [0, limite], color="cyan", linestyle="--", linewidth=1.1)
    ax.set(
        title=f"{titulo}\naté o percentil {PERCENTIL_LIMITE_HEATMAP:g}",
        xlabel=f"Distância euclidiana {rotulo}",
        ylabel=f"Distância {rotulo_distancia_agregado} {rotulo}",
        xlim=(0, limite), ylim=(0, limite),
    )
    ax.set_aspect("equal", adjustable="box")
    barra = plt.colorbar(mapa, ax=ax)
    barra.set_label("Pares amostrados por hexágono (escala log)")
    return mapa

fig_heatmap, eixos_heatmap = plt.subplots(
    1, 2, figsize=(14, 6.2), constrained_layout=True
)
desenhar_heatmap_distancias(
    eixos_heatmap[0], euclid_heatmap, mrd_heatmap,
    "A. Todas as runs — distâncias brutas", "",
)
desenhar_heatmap_distancias(
    eixos_heatmap[1], euclid_normalizada, mrd_normalizada,
    "B. Todas as runs — escala comparável",
    "/ mediana euclidiana da run",
)
fig_heatmap.suptitle(
    f"Densidade Euclidiana × {rotulo_distancia_agregado} | {len(amostras_heatmap)} runs | "
    f"{len(euclid_heatmap):,} pares",
    fontsize=14,
)

if SALVAR_AGREGADO:
    pasta_agregada = resolver(PASTA_SAIDA_AGREGADA)
    pasta_agregada.mkdir(parents=True, exist_ok=True)
    fig_heatmap.savefig(
        pasta_agregada / "all_runs_distance_density_heatmap.png",
        dpi=180, bbox_inches="tight",
    )
plt.show()

### Leitura do heatmap agregado

- A linha tracejada representa MRD/MMSRD igual à distância euclidiana.
- Regiões mais claras contêm mais pares nó–cluster.
- A concentração acima da diagonal mostra onde a MRD ou MMSRD ampliou a distância.
- O painel bruto preserva a escala original dos embeddings.
- O painel normalizado divide as duas distâncias pela mediana euclidiana de cada run, evitando que uma execução com embeddings numericamente maiores determine os eixos.
- Cada run fornece no máximo PARES_HEATMAP_POR_RUN pares. Assim, execuções com mais clusters não dominam a densidade apenas por terem mais observações.

## Como interpretar a análise agregada

- Cada ponto representa uma run completa; uma execução com mais clusters não recebe peso adicional.
- O painel A mede com que frequência a MRD/MMSRD deixa de ser numericamente igual à distância euclidiana.
- O painel B mede o efeito sobre a decisão dura de cluster.
- O painel C revela se mais distâncias alteradas também implicam mais vértices reatribuídos.
- O painel D resume a intensidade da alteração pelo percentil 95 do aumento relativo.
- partition_ari e partition_nmi na tabela comparam as duas partições no mesmo embedding; valores próximos de 1 indicam decisões semelhantes.
- Esta continua sendo uma análise pós-treinamento. Para medir impacto causal sobre NMI, modularidade ou silhouette, são necessárias runs pareadas treinadas desde o início com MRD e Euclidiana.

## Como interpretar

- Pontos na diagonal indicam pares nó–cluster nos quais as core distances não alteraram a distância euclidiana.
- Pontos acima da diagonal foram ampliados pela MRD ou MMSRD, conforme o modelo selecionado.
- Um aumento de distância não implica automaticamente uma melhoria de clustering.
- A quantidade de nós que troca de cluster mede o efeito da distância sobre a decisão final naquele embedding.
- Mesmo quando poucos nós mudam, as atribuições suaves Q, as margens e os gradientes durante o treinamento podem ter sido diferentes.
- Para afirmar que uma distância é melhor, compare runs treinadas desde o início com as mesmas seeds e configurações, usando métricas e diferenças pareadas.